# wandb-config-into-args — worked example 1: Apply a sweep config dict onto an args dataclass

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-config-into-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

During a wandb sweep, the sweep agent calls your training function with hyperparameters stored in `wandb.config`. The canonical pattern to apply them is to iterate over `dict(wandb.config).items()` and use `setattr(args, k, v)` for each key that exists on the args dataclass, silently skipping unknown keys (like wandb's internal `_wandb` metadata).

## Worked solution

**Step 1 — understand the inputs.**
The `args` dataclass already has all hyperparameter fields with default values. The `sweep_cfg` dict (from `dict(wandb.config)`) contains the values the sweep agent sampled — but it may also include wandb-internal keys like `'_wandb'` or `'_runtime'` that are not fields on your dataclass.

**Step 2 — iterate and guard.**
We loop over `sweep_cfg.items()`. For each key `k`, we check `hasattr(args, k)`. This guard ensures we only write fields that actually exist on the dataclass — unknown wandb metadata keys are silently skipped rather than raising an AttributeError.

**Step 3 — overwrite with setattr.**
For valid keys, we call `setattr(args, k, v)`. This replaces the default value with the sweep-sampled value in place. We return the same (mutated) args object.

**Why this matters.**
Without this pattern, every sweep trial would use the dataclass defaults regardless of what the sweep agent sampled, making the sweep pointless.

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass

sys.modules.setdefault('wandb', MagicMock())
import wandb

@dataclass
class TrainArgs:
    lr: float = 1e-3
    batch_size: int = 64
    dropout: float = 0.1
    wandb_project: str = 'my-project'

def apply_sweep_config(args, sweep_cfg):
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args

# Exercise it
args = TrainArgs()
print('Before:', args)

sweep_cfg = {'lr': 5e-4, 'batch_size': 128, '_wandb': {'runtime': 12}, '_runtime': 5}
apply_sweep_config(args, sweep_cfg)
print('After:', args)
print('lr updated to:', args.lr)  # 5e-4
print('batch_size updated to:', args.batch_size)  # 128